___


___

# Vocabulario y matching (comparador)
Hasta ahora hemos visto cómo un cuerpo de texto se divide en tokens y cómo los tokens individuales se analizan y etiquetan con parts of speech, dependencias y lemas.

En esta sección identificaremos y etiquetaremos frases específicas que coincidan con patrones que podemos definir nosotros mismos.

## Coincidencia basada en reglas
spaCy ofrece una herramienta de coincidencia de reglas llamada `Matcher` que le permite crear una biblioteca de patrones de token y luego comparar esos patrones con un objeto Doc para devolver una lista de coincidencias encontradas. Puede hacer coincidir cualquier parte del token, incluido el texto y las anotaciones, y puede agregar varios patrones al mismo comparador.

In [1]:
# Realizar importaciones estándar
import spacy
nlp = spacy.load('en_core_web_sm')

In [2]:
# Importar la biblioteca Matcher
from spacy.matcher import Matcher
matcher = Matcher(nlp.vocab)

<font color=green>Aquí `matcher` es un objeto que se empareja con el objeto `Vocab` actual. Podemos agregar y eliminar comparadores con nombre específicos a `matcher` según sea necesario.</font>

### Creando patrones
En la literatura, la frase "solar power" puede aparecer como una palabra o dos, con o sin guión. En esta sección, desarrollaremos un comparador llamado 'SolarPower' que encuentra los tres:

In [3]:
pattern1 = [{'LOWER': 'solarpower'}]
pattern2 = [{'LOWER': 'solar'}, {'LOWER': 'power'}]
pattern3 = [{'LOWER': 'solar'}, {'IS_PUNCT': True}, {'LOWER': 'power'}]

matcher.add('SolarPower', None, pattern1, pattern2, pattern3)

Desglosemos esto:
* `patrón1` busca un solo token cuyo texto en minúsculas diga 'solarpower'
* `patrón2` busca dos tokens adyacentes que digan 'solar' y 'power' en ese orden
* `patrón3` busca tres tokens adyacentes, con un token en el medio que puede ser cualquier puntuación.<font color=green>*</font>

<font color=green>\* Recuerde que los espacios individuales no se tokenizan, por lo que no cuentan como puntuación.</font>
<br>Una vez que definimos nuestros patrones, los pasamos a `matcher` con el nombre 'SolarPower' y configuramos *callbacks* en `None` (más información sobre callbacks más adelante).

### Aplicar el matcher (comparador) a un objeto Doc

In [5]:
doc = nlp(u'The Solar Power industry continues to grow as demand \
for solarpower increases. Solar-power cars are gaining popularity.')

In [6]:
found_matches = matcher(doc)
print(found_matches)

[(8656102463236116519, 1, 3), (8656102463236116519, 10, 11), (8656102463236116519, 13, 16)]


`matcher` devuelve una lista de tuplas. Cada tupla contiene una ID para la coincidencia, con tokens de inicio y finalización que se asignan al intervalo `doc[start:end]`

In [7]:
for match_id, start, end in found_matches:
    string_id = nlp.vocab.strings[match_id]  # obtener representación de cadena
    span = doc[start:end]                    # obtener el span coincidente
    print(match_id, string_id, start, end, span.text)

8656102463236116519 SolarPower 1 3 Solar Power
8656102463236116519 SolarPower 10 11 solarpower
8656102463236116519 SolarPower 13 16 Solar-power


El `match_id` es simplemente el valor hash del `string_ID` 'SolarPower'

### Configuración de opciones de patrón y cuantificadores
Puede hacer que las reglas de token sean opcionales pasando un argumento `'OP':'*'`. Esto nos permite simplificar nuestra lista de patrones:

In [8]:
# Redefine los patrones:
pattern1 = [{'LOWER': 'solarpower'}]
pattern2 = [{'LOWER': 'solar'}, {'IS_PUNCT': True, 'OP':'*'}, {'LOWER': 'power'}]

# Elimine los patrones antiguos para evitar la duplicación:
matcher.remove('SolarPower')

# Agregue el nuevo conjunto de patrones al comparador 'SolarPower':
matcher.add('SolarPower', None, pattern1, pattern2)

In [9]:
found_matches = matcher(doc)
print(found_matches)

[(8656102463236116519, 1, 3), (8656102463236116519, 10, 11), (8656102463236116519, 13, 16)]


¡Esto encontró ambos patrones de dos palabras, con y sin el guión!

Los siguientes cuantificadores se pueden pasar a la tecla `'OP'`:
<table><tr><th>OP</th><th>Descripción</th></tr>

<tr ><td><span >\!</span></td><td>Niega el patrón, requiriendo que coincida exactamente 0 veces</td></tr>
<tr ><td><span >?</span></td><td>Haga que el patrón sea opcional, permitiendo que coincida 0 o 1 veces</td></tr>
<tr ><td><span >\+</span></td><td>Requerir que el patrón coincida 1 o más veces</td></tr>
<tr ><td><span >\*</span></td><td>Permitir que el patrón coincida cero o más veces</td></tr>
</tabla>


### ¡Cuidado con los lemas!
Si quisiéramos hacer coincidir tanto 'solar power' como 'solar powered', sería tentador buscar el *lema* de 'powered' y esperar que fuera 'power'. ¡Este no es siempre el caso! El lema del *adjetivo* 'powered' sigue siendo 'powered':

In [10]:
pattern1 = [{'LOWER': 'solarpower'}]
pattern2 = [{'LOWER': 'solar'}, {'IS_PUNCT': True, 'OP':'*'}, {'LEMMA': 'power'}] # Cambia este patrón

# Elimine los patrones antiguos para evitar la duplicación:
matcher.remove('SolarPower')

# Agregue el nuevo conjunto de patrones al matcher 'SolarPower':
matcher.add('SolarPower', None, pattern1, pattern2)

In [11]:
doc2 = nlp(u'Solar-powered energy runs solar-powered cars.')

In [12]:
found_matches = matcher(doc2)
print(found_matches)

[(8656102463236116519, 0, 3)]


<font color=green>El matcher encontró la primera ocurrencia porque el lematizador trató 'Solar-powered' como un verbo, pero no la segunda, ya que lo consideró un adjetivo.<br>Para este caso, puede ser mejor establecer un token explícito patrones.</font>

In [13]:
pattern1 = [{'LOWER': 'solarpower'}]
pattern2 = [{'LOWER': 'solar'}, {'IS_PUNCT': True, 'OP':'*'}, {'LOWER': 'power'}]
pattern3 = [{'LOWER': 'solarpowered'}]
pattern4 = [{'LOWER': 'solar'}, {'IS_PUNCT': True, 'OP':'*'}, {'LOWER': 'powered'}]

# Elimine los patrones antiguos para evitar la duplicación:
matcher.remove('SolarPower')

# Agregue el nuevo conjunto de patrones al matcher 'SolarPower':
matcher.add('SolarPower', None, pattern1, pattern2, pattern3, pattern4)

In [14]:
found_matches = matcher(doc2)
print(found_matches)

[(8656102463236116519, 0, 3), (8656102463236116519, 5, 8)]


## Otros atributos del token
Además de los lemas, hay una variedad de atributos de token que podemos usar para determinar las reglas de coincidencia:
<table><tr><th>Atributo</th><th>Descripción</th></tr>

<tr ><td><span >`ORTH`</span></td><td>El texto textual exacto de un token</td></tr>
<tr ><td><span >`LOWER`</span></td><td>La forma en minúsculas del texto del token</td></tr>
<tr ><td><span >`LENGTH`</span></td><td>La longitud del texto del token</td></tr>
<tr ><td><span >`IS_ALPHA`, `IS_ASCII`, `IS_DIGIT`</span></td><td>El texto del token consiste en caracteres alfanuméricos, caracteres ASCII, dígitos</td></tr>
<tr ><td><span >`IS_LOWER`, `IS_UPPER`, `IS_TITLE`</span></td><td>El texto del token está en minúsculas, mayúsculas y título</td></tr>
<tr ><td><span >`IS_PUNCT`, `IS_SPACE`, `IS_STOP`</span></td><td>El token es puntuación, espacios en blanco, palabras vacías</td></tr>
<tr ><td><span >`LIKE_NUM`, `LIKE_URL`, `LIKE_EMAIL`</span></td><td>El texto del token se asemeja a un número, URL, correo electrónico</td></tr>
<tr ><td><span >`POS`, `TAG`, `DEP`, `LEMMA`, `SHAPE`</span></td><td>La etiqueta de parte del discurso simple y extendida del token , etiqueta de dependencia, lema, forma</td></tr>
<tr ><td><span >`ENT_TYPE`</span></td><td>La etiqueta de entidad del token</td></tr>

</tabla>

### Token comodín
Puede pasar un diccionario vacío `{}` como comodín para representar **cualquier token**. Por ejemplo, es posible que desee recuperar hashtags sin saber qué puede seguir al carácter `#`:
>`[{'ORTO': '#'}, {}]`

___
## Combinador de frases
En la sección anterior, usamos patrones de token para realizar coincidencias basadas en reglas. Un método alternativo, y a menudo más eficiente, es hacer coincidir las listas de terminología. En este caso, usamos PhraseMatcher para crear un objeto Doc a partir de una lista de frases y, en su lugar, lo pasamos a `matcher`.

In [15]:
# Realizar importaciones estándar, restablecer nlp
import spacy
nlp = spacy.load('en_core_web_sm')

In [16]:
# Importar la biblioteca de PhraseMatcher
from spacy.matcher import PhraseMatcher
matcher = PhraseMatcher(nlp.vocab)

Para este ejercicio vamos a importar un artículo de Wikipedia sobre *Reaganomics*<br>
Fuente: https://en.wikipedia.org/wiki/Reaganomics

In [19]:
with open('../datos/reaganomics.txt', encoding='utf8') as f:
    doc3 = nlp(f.read())

In [20]:
# Primero, cree una lista de frases coincidentes:
phrase_list = ['voodoo economics', 'supply-side economics', 'trickle-down economics', 'free-market economics']

# A continuación, convierta cada frase en un objeto Doc:
phrase_patterns = [nlp(text) for text in phrase_list]

# Pase cada objeto Doc al comparador (¡tenga en cuenta el uso del asterisco para desempaquetar la lista!):
matcher.add('VoodooEconomics', None, *phrase_patterns)

# Cree una lista de coincidencias:
matches = matcher(doc3)

In [21]:
# (match_id, start, end)
matches

[(3473369816841043438, 41, 45),
 (3473369816841043438, 49, 53),
 (3473369816841043438, 54, 56),
 (3473369816841043438, 61, 65),
 (3473369816841043438, 673, 677),
 (3473369816841043438, 2984, 2988)]

<font color=green>Las primeras cuatro coincidencias son donde se usan estos términos en la definición de Reaganomics:</font>

In [22]:
doc3[:70]

REAGANOMICS
https://en.wikipedia.org/wiki/Reaganomics

Reaganomics (a portmanteau of [Ronald] Reagan and economics attributed to Paul Harvey)[1] refers to the economic policies promoted by U.S. President Ronald Reagan during the 1980s. These policies are commonly associated with supply-side economics, referred to as trickle-down economics or voodoo economics by political opponents, and free-market economics by political advocates.


## Visualización de matches
Hay algunas formas de obtener el texto que rodea una coincidencia. El más simple es tomar una porción de fichas del documento que sea más ancha que la coincidencia:

In [23]:
doc3[665:685]  # Tenga en cuenta que la quinta coincidencia comienza en doc3[673]

same time he attracted a following from the supply-side economics movement, which formed in opposition to Keynesian

In [24]:
doc3[2975:2995]  # El sexto match comienza en doc3[2985]

institutions.[66] His policies became widely known as "trickle-down economics", due to the significant cuts

Otra forma es aplicar primero el `sentencizer` al Doc, luego iterar a través de las oraciones hasta el punto de coincidencia:

In [25]:
# Construye una lista de oraciones.
sents = [sent for sent in doc3.sents]

# En la siguiente sección, veremos que las oraciones contienen valores de token de inicio y fin:
print(sents[0].start, sents[0].end)

0 35


In [26]:
# Iterar sobre la lista de oraciones hasta que el valor final de la oración exceda el valor inicial de la coincidencia:
for sent in sents:
    if matches[4][1] < sent.end:  # esta es la quinta coincidencia, que comienza en doc3[673]
        print(sent)
        break

At the same time he attracted a following from the supply-side economics movement, which formed in opposition to Keynesian demand-stimulus economics.


Para obtener información adicional, visite https://spacy.io/usage/linguistic-features#section-rule-based-matching
## A continuación: Taller de conceptos básicos de NLP